# Lab type: debug
# Course: ML401 — MLOps & Model Deployment
# Lesson: Model Packaging and Serialisation
# Task: The model saving and loading pipeline below has 3 bugs that will cause silent failures in production. Identify each bug, explain the failure mode, and write the corrected code.

In [ ]:
# Install dependencies (Colab)
# !pip install scikit-learn joblib pandas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import joblib
import json
import os

# Generate a synthetic classification dataset
X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    random_state=42
)
feature_names = [f'feature_{i}' for i in range(10)]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)
print(f'Training samples: {len(X_train)}, Test samples: {len(X_test)}')

## The broken packaging pipeline

The following code trains a model and saves it for production deployment.
There are **3 bugs** that will cause silent failures when this model is loaded in a production environment.

Read the code carefully. Do not run it yet — first identify all three bugs.

In [ ]:
# === BUGGY CODE — DO NOT USE IN PRODUCTION ===

# Bug hunt: find 3 production failures before running this

# Train the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)

# Save — Bug 1 is here
joblib.dump(pipeline.named_steps['model'], 'model_artifact.joblib')

# Save metadata — Bug 2 is here
metadata = {
    'description': 'churn prediction model',
    'accuracy': float((pipeline.predict(X_test) == y_test).mean())
    # no version information
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)

print('Model saved.')

# === INFERENCE CODE (runs in a different environment) ===
# Bug 3 is here
loaded_model = joblib.load('model_artifact.joblib')

# Simulate inference: new data arrives in a different column order
X_inference = X_test[sorted(X_test.columns)]  # columns reordered alphabetically
predictions = loaded_model.predict(X_inference)
print(f'Predictions: {predictions[:5]}')

## Your analysis

Before looking at the answers, write your analysis here.

**Bug 1** (saving step):
- What is wrong:
- What failure it causes in production:

**Bug 2** (metadata step):
- What is wrong:
- What failure it causes in production:

**Bug 3** (inference step):
- What is wrong:
- What failure it causes in production:

## Answers

**Bug 1:** `pipeline.named_steps['model']` saves only the LogisticRegression classifier — not the StandardScaler. At inference time, raw unscaled features are passed directly to the classifier, which was trained on scaled inputs. The model runs and produces predictions, but on out-of-distribution inputs. The StandardScaler's learned mean and standard deviation are lost.

**Bug 2:** The metadata does not record the library versions (Python, scikit-learn) or the expected feature schema (names, count, order). When this model is loaded in a different environment six months later, there is no contract to validate. A version mismatch or feature schema change produces wrong predictions with no warning.

**Bug 3:** The inference code loads only the classifier (not the pipeline). Even if the full pipeline were saved, the inference code reorders the columns alphabetically before predicting. The model was trained on a specific column order; reordering the columns maps features to wrong positions. For a LogisticRegression, this means each coefficient is multiplied against the wrong feature.

In [ ]:
import sys
import sklearn
from datetime import datetime

# === CORRECTED CODE ===

# Fix 1: Save the ENTIRE pipeline, not just the classifier
joblib.dump(pipeline, 'pipeline_artifact.joblib')

# Fix 2: Record version contract and feature schema
metadata_correct = {
    'description': 'churn prediction model',
    'saved_at': datetime.utcnow().isoformat(),
    'python_version': sys.version,
    'sklearn_version': sklearn.__version__,
    'feature_names': list(X_train.columns),
    'n_features': len(X_train.columns),
    'accuracy_test': float((pipeline.predict(X_test) == y_test).mean())
}
with open('pipeline_metadata.json', 'w') as f:
    json.dump(metadata_correct, f, indent=2)

print('Contract saved:')
print(json.dumps(metadata_correct, indent=2))

# Fix 3: At inference time, reorder columns to match training order using the contract
with open('pipeline_metadata.json') as f:
    contract = json.load(f)

loaded_pipeline = joblib.load('pipeline_artifact.joblib')

# Validate and reorder columns to match training order
expected_features = contract['feature_names']
assert set(X_inference.columns) == set(expected_features), 'Feature set mismatch'
X_inference_ordered = X_inference[expected_features]  # enforce training column order

predictions_correct = loaded_pipeline.predict(X_inference_ordered)
print(f'Correct predictions: {predictions_correct[:5]}')

## Reflection questions

1. If you added a new feature to the model and redeployed without updating the contract, what would happen at inference time with the corrected code above?

2. MLflow automates the version contract (Python version, library versions, feature schema). What would you lose by switching from this manual contract approach to MLflow? What would you gain?

3. The corrected code still overwrites `pipeline_artifact.joblib` on each run. How would you modify it to maintain a versioned artefact history?